In [1]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
#pip install pycbc
from pycbc.psd.analytical import aLIGO175MpcT1800545
from scipy.integrate import odeint

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

/home/alberto/ns-resonance/jupyter-env/lib/python3.12/site-packages/pycbc/types/array.py:36: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal as _lal
PyCBC.libutils: pkg-config call failed, setting NO_PKGCONFIG=1


In [2]:
import sys
import os

sys.path.insert(1, '../../detectability')

import resonance
from resonance import System, Filter, phase_diff_t_shift, phase_shift, dEdf_box, extra_flux_box, solve_ode,jultosec

In [3]:
data_path = '../data/mismatch_plots_data/quad'
os.makedirs(data_path, exist_ok=True) 

# Inputs

In [4]:
m2=1.4
m1=1.4 #3*1.4=4.2


filt = Filter(15, 1024, 60, 16384, 100, 'IMRPhenomD', 'aLIGO/AplusDesign', m1, m2) #f_low, f_high, tlen, srate, dL, approximant, detector, m1, m2

binary = System(m1, m2)

Requested number of samples exceeds the highest available frequency in the input data, will use max available frequency instead. (requested 8192.000000 Hz, available 5000.000000 Hz)
/home/alberto/ns-resonance/jupyter-env/lib/python3.12/site-packages/pycbc/types/array.py:390: RuntimeWarning: divide by zero encountered in divide
  return self._data.__rtruediv__(other)


In [5]:
m1

1.4

# Obtain data grid

In [6]:
len_arr = 21
fres_arr = np.linspace(30., 500., len_arr)
dt_arr = -np.logspace(-4, -2.5, len_arr) # dt < 0

fres_grid, dt_grid = np.meshgrid(fres_arr, dt_arr)

In [7]:
#error = 1e-3

#dt_cut_arr = filt.dx_fres_fix_error(error, fres_arr, phase_diff_t_shift)
#print(dt_cut_arr)

In [8]:
def MM_dF(fres, dt):
    print(fres, dt)
    return 1 - filt.match_approx(phase_diff_t_shift(dt, fres, filt.freqs)), binary.dE(dt, fres), binary.dF(dt, fres)/binary.flux(fres)

In [9]:
MM_dE_vec = np.vectorize(MM_dF)

In [10]:
MM, dE, dF = MM_dE_vec(fres_grid, dt_grid)

30.0 -0.0001
30.0 -0.0001
53.5 -0.0001
77.0 -0.0001
100.5 -0.0001
124.0 -0.0001
147.5 -0.0001
171.0 -0.0001
194.5 -0.0001
218.0 -0.0001
241.5 -0.0001
265.0 -0.0001
288.5 -0.0001
312.0 -0.0001
335.5 -0.0001
359.0 -0.0001
382.5 -0.0001
406.0 -0.0001
429.5 -0.0001
453.0 -0.0001
476.5 -0.0001
500.0 -0.0001
30.0 -0.00011885022274370189
53.5 -0.00011885022274370189
77.0 -0.00011885022274370189
100.5 -0.00011885022274370189
124.0 -0.00011885022274370189
147.5 -0.00011885022274370189
171.0 -0.00011885022274370189
194.5 -0.00011885022274370189
218.0 -0.00011885022274370189
241.5 -0.00011885022274370189
265.0 -0.00011885022274370189
288.5 -0.00011885022274370189
312.0 -0.00011885022274370189
335.5 -0.00011885022274370189
359.0 -0.00011885022274370189
382.5 -0.00011885022274370189
406.0 -0.00011885022274370189
429.5 -0.00011885022274370189
453.0 -0.00011885022274370189
476.5 -0.00011885022274370189
500.0 -0.00011885022274370189
30.0 -0.0001412537544622754
53.5 -0.0001412537544622754
77.0 -0.00014

In [11]:
np.savez_compressed(os.path.join(data_path, f'MM_dE_{m1}.npz'), columns=['fres','dt','MM','dE','dF'], data=[fres_grid, dt_grid, MM, dE/jultosec*1e7, dF]) #energy in erg

In [12]:
#np.savez_compressed(os.path.join(data_path, f'cut_{m1}.npz'), columns=['fres','dt_cut'], data=[fres_arr,dt_cut_arr]) #energy in erg